# 1.2 — Fit the generic assistant as a mixture of persona components (OLMo 3 base)

**The Phase 1 measurement.** Responses sampled from the generic prompt $P_0$ (the `unknown` framing:
an assistant with a well-defined but a-priori-unknown character) on the temptation-dilemma question
set, scored under the five description-only components. We fit the mixture weights $w_s$ by EM on half
the questions and report the held-out KL $\hat D = \frac{1}{N}\sum_i[\log P_0(a_i) - \log\sum_s w_s
P(a_i\mid s)]$ on the other half, with bootstrap error bars over questions.

Reference values: the best single component ($K=1$), the full sweep over subsets, and the calibration
floor from 1.1. Then the sanity checks that matter: what do the evil-assigned samples actually say, and
how do the fitted weights compare with the model's own self-reported prior (0.9) and the uniform-prior
assignment in 0.10?

Data from `scripts/phase1_sample_score.py --source generic --framing unknown`.

In [ ]:
import os, sys, json, textwrap, collections
from pathlib import Path
import numpy as np
from scipy.special import softmax, logsumexp
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from persona_selection.mixture import em_weights, kl_estimate, fit_and_evaluate, bootstrap_over_groups, k_sweep

RUN = REPO / "results" / "phase1" / "base_unknown_v1"
z = np.load(RUN / "matrix.npz", allow_pickle=True)
rows = [json.loads(l) for l in open(RUN / "rows.jsonl")]
cfg = json.loads((RUN / "config.json").read_text())
L, l0, n_tok, groups = z["L"], z["l0"], z["n_tokens"], z["groups"]
names = list(z["personas"].astype(str))
print(f"{len(rows)} responses from {len(set(groups))} questions | components {names} | model {cfg['args']['model']} | framing {cfg['args']['framing']}")
print(f"median response length {np.median(n_tok):.0f} tokens")
print("\ngeneric prompt:\n" + cfg["generic_prompt_example"])

## The fit

In [ ]:
w_full, info = em_weights(L)
kl_full = kl_estimate(l0, L, w_full, n_tok)
r = fit_and_evaluate(L, l0, groups, n_tok)
b = bootstrap_over_groups(L, l0, groups, n_boot=300, n_tokens=n_tok)
print(f"{'component':>10} {'w (all data)':>13} {'w (split fit)':>14} {'bootstrap sd':>13} {'boot 5%':>8} {'boot 95%':>9}")
for i, n in enumerate(names):
    lo, hi = np.percentile(b["w"][:, i], [5, 95])
    print(f"{n:>10} {w_full[i]:13.3f} {r['w'][i]:14.3f} {b['w'][:, i].std():13.3f} {lo:8.3f} {hi:9.3f}")
print(f"\nheld-out KL: {r['heldout']['kl_per_response']:+.3f} ± {r['heldout']['kl_se']:.3f} nats/response "
      f"({r['heldout']['kl_per_token']:+.4f}/token); bootstrap mean {b['kl_heldout'].mean():+.3f}, sd {b['kl_heldout'].std():.3f}")
print(f"in-sample KL (all data): {kl_full['kl_per_response']:+.3f} nats/response")
print(f"\nFor scale: mean log P_0 per response = {l0.mean():.1f} nats; mean best-component log-lik = {L.max(1).mean():.1f}; "
      f"median (log P_0 - best component) = {np.median(l0 - L.max(1)):+.2f}")

## K sweep: how many components does the generic assistant need?

Held-out KL for the best subset of each size. The $K=1$ row is the "single persona" reference the
README asks for. If the curve flattens well above the calibration floor, the generic assistant contains
something none of these five components captures (or the components are too far from it in style).

In [ ]:
sw = k_sweep(L, l0, groups, names, n_tok)
best = {}
for row in sw:
    if row["k"] not in best or row["kl_heldout"] < best[row["k"]]["kl_heldout"]:
        best[row["k"]] = row
for k, row in sorted(best.items()):
    print(f"K={k}: {row['subset']!s:50} held-out KL {row['kl_heldout']:+.3f} ± {row['kl_heldout_se']:.3f}   w={ {n: round(x, 3) for n, x in row['w'].items()} }")
print("\nall single components:")
for row in sw:
    if row["k"] == 1:
        print(f"  {row['subset'][0]:>10}: held-out KL {row['kl_heldout']:+.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(names))
ax1.bar(x, w_full, color="#2C6FB3", yerr=b["w"].std(0), capsize=3)
ax1.set_xticks(x); ax1.set_xticklabels(names); ax1.set_ylabel("fitted mixture weight"); ax1.set_title("Mixture weights (EM, bootstrap sd)")
ks = sorted(best); ax2.errorbar(ks, [best[k]["kl_heldout"] for k in ks], yerr=[best[k]["kl_heldout_se"] for k in ks], marker="o", color="#2C6FB3", capsize=3)
ax2.axhline(0, color="0.5", lw=0.8); ax2.set_xlabel("number of components K"); ax2.set_ylabel("held-out KL [nats / response]"); ax2.set_title("Best subset per K")
for ax in (ax1, ax2): ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); fig.savefig(REPO / "results" / "phase1" / "1.2_base_fit.png", dpi=150); plt.show()

## Responsibilities: what did the fit assign to the non-HHH components, and are those samples real?

Posterior responsibilities $\gamma_{is}$ under the fitted weights (not the uniform prior of 0.10).
Read the samples with the highest evil responsibility: misaligned content, junk, or style?

In [ ]:
gamma = info["gamma"]
hard = collections.Counter(names[i] for i in gamma.argmax(1))
print("hard assignment under fitted weights: " + ", ".join(f"{n} {100*hard[n]/len(rows):.0f}%" for n in names))
print("mean responsibility:                " + ", ".join(f"{n} {100*gamma[:, i].mean():.0f}%" for i, n in enumerate(names)))
ei = names.index("evil")
order = np.argsort(-gamma[:, ei])
print(f"\ntop {12} samples by evil responsibility:")
for i in order[:12]:
    print(f"  γ_evil={gamma[i, ei]:.2f} | Q: {rows[i]['question'][:50]:50} | {textwrap.shorten(rows[i]['response'].strip(), 130)}")
print("\nper-category evil responsibility (mean):")
cats = collections.defaultdict(list)
for i, rr in enumerate(rows):
    cats[rr["category"]].append(gamma[i, ei])
for c, v in sorted(cats.items(), key=lambda kv: -np.mean(kv[1])):
    print(f"  {c:>18}: {100*np.mean(v):5.1f}%  (n={len(v)})")

## Against the self-report (0.9) and the uniform-prior view (0.10)

The scored-candidate prior (0.9, `corpus` framing) and the direct question to the instruct model
both put `evil` far below 1%. 0.10's uniform-prior argmax put it near 15–18%. The fitted weight is the
behavioural number those were meant to predict.

In [ ]:
try:
    prior = json.load(open(REPO / "results" / "phase0" / "0.9_persona_prior.json"))
    A = prior["scored_candidates"]["allenai/Olmo-3-1025-7B"]["corpus"]["p"]
    lookup = {"hhh": "Helpful Assistant", "evil": "Evil Assistant", "sycophant": "Supportive Assistant", "fred": "Cynical Assistant", "formal": "Formal Assistant"}
    print(f"{'component':>10} {'fitted w':>9} {'self-report (0.9 corpus, nearest label)':>42}")
    for i, n in enumerate(names):
        print(f"{n:>10} {w_full[i]:9.3f} {100*A.get(lookup[n], float('nan')):40.2f}%  [{lookup[n]}]")
except FileNotFoundError:
    print("0.9 results not found; skipping")

In [ ]:
json.dump({"run": str(RUN), "components": names, "w_full": w_full.tolist(), "w_split": r["w"].tolist(), "boot_w_sd": b["w"].std(0).tolist(),
           "heldout": r["heldout"], "boot_kl_mean": float(b["kl_heldout"].mean()), "boot_kl_sd": float(b["kl_heldout"].std()),
           "ksweep_best": {str(k): v for k, v in best.items()}, "hard_assignment": dict(hard),
           "mean_responsibility": dict(zip(names, gamma.mean(0).tolist()))},
          open(REPO / "results" / "phase1" / "1.2_base_fit.json", "w"), indent=2)
print("saved results/phase1/1.2_base_fit.{png,json}")

## What to look for

- **Held-out KL vs. the calibration floor (1.1).** Near the floor: the generic assistant *is* well
  described as a mixture of these components (the PSM-consistent outcome). Far above: something in the
  generic samples is not captured by any component; check whether the junk fraction (broken turns,
  off-topic drift) explains it before concluding anything about personas.
- **K sweep shape.** Does the second component help much? Which one is it? That is the answer to "how
  many personas does the base assistant contain" for this basis.
- **Evil weight vs self-report.** A fitted `evil` weight of several percent, with the top-responsibility
  samples genuinely selfish, would be a clean disagreement between what the model *says* its prior is
  and what it *does*. A weight near zero, with the 0.10 assignment explained by junk, would reconcile them.
- Everything here is one framing (`unknown`) and one basis (five components). The sensitivity runs
  (`story`, `minimal`; more components) are the next step, not this notebook.